# FleetManager — Analyse de la flotte

**Auteur** : Raphaël Quilichini  
**Module** : Outils technologiques pour la digitalisation (ESSCA)  
**Contexte métier** : pilotage du TCO (Total Cost of Ownership) d'une flotte automobile.

Ce notebook répond à la phase **Visualisation (07/05)** : agrégations, regroupements, extrêmes, comptages, accompagnés d'une analyse métier.

## ⚙️ Exécution

Ce notebook utilise le kernel **rapaio-jupyter-kernel** (Java in JShell).

Pour reproduire les résultats sans Jupyter, exécuter en ligne de commande :

```
javac *.java
java MainAnalyse
```

Tous les chiffres et observations métier ci-dessous ont été générés par cette exécution Java (voir `resultats_analyse.txt`), puis intégrés et commentés dans ce notebook.


## 1.Chargement des classes ##

In [ ]:
import java.util.ArrayList;
import java.util.List;

// ============================================================
// TypeEntretien
// ============================================================
class TypeEntretien {
    private String idTypeEntretien;
    private String libelle;
    private List<Entretien> entretiens;

    public TypeEntretien(String idTypeEntretien, String libelle) {
        this.entretiens = new ArrayList<>();
        setIdTypeEntretien(idTypeEntretien);
        setLibelle(libelle);
    }
    public TypeEntretien(String[] champs) { this(champs[0], champs[1]); }

    public String getIdTypeEntretien() { return idTypeEntretien; }
    public String getLibelle()         { return libelle; }
    public List<Entretien> getEntretiens() { return entretiens; }

    public void setIdTypeEntretien(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idTypeEntretien obligatoire");
        this.idTypeEntretien = v;
    }
    public void setLibelle(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("libelle obligatoire");
        this.libelle = v;
    }
    public void addEntretien(Entretien e) { if (e != null) entretiens.add(e); }
}

// ============================================================
// TypeEnergie
// ============================================================
class TypeEnergie {
    private String idTypeEnergie;
    private String libelle;
    private String unite;
    private double prixParUnite;
    private List<Vehicule> vehicules;

    public TypeEnergie(String idTypeEnergie, String libelle, String unite, double prixParUnite) {
        this.vehicules = new ArrayList<>();
        setIdTypeEnergie(idTypeEnergie);
        setLibelle(libelle);
        setUnite(unite);
        setPrixParUnite(prixParUnite);
    }
    public TypeEnergie(String[] champs) {
        this(champs[0], champs[1], champs[2], Double.parseDouble(champs[3].trim()));
    }

    public String getIdTypeEnergie() { return idTypeEnergie; }
    public String getLibelle()       { return libelle; }
    public String getUnite()         { return unite; }
    public double getPrixParUnite()  { return prixParUnite; }
    public List<Vehicule> getVehicules() { return vehicules; }

    public void setIdTypeEnergie(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idTypeEnergie obligatoire");
        this.idTypeEnergie = v;
    }
    public void setLibelle(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("libelle obligatoire");
        this.libelle = v;
    }
    public void setUnite(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("unite obligatoire");
        this.unite = v;
    }
    public void setPrixParUnite(double v) {
        if (v < 0) throw new IllegalArgumentException("prixParUnite >= 0");
        this.prixParUnite = v;
    }
    public void addVehicule(Vehicule v) { if (v != null) vehicules.add(v); }
}

// ============================================================
// Vehicule
// ============================================================
class Vehicule {
    private String idVehicule;
    private String marque;
    private String modele;
    private String idTypeEnergie;
    private double prixAchat;
    private double consommation;
    private int autonomieKm;
    private int anneeAchat;
    private int dureeAmortissement;

    private TypeEnergie typeEnergie;
    private List<Entretien> entretiens;
    private List<UtilisationAnnuelle> utilisationsAnnuelles;

    public Vehicule(String idVehicule, String marque, String modele, String idTypeEnergie,
                    double prixAchat, double consommation, int autonomieKm,
                    int anneeAchat, int dureeAmortissement) {
        this.entretiens = new ArrayList<>();
        this.utilisationsAnnuelles = new ArrayList<>();
        setIdVehicule(idVehicule);
        setMarque(marque);
        setModele(modele);
        setIdTypeEnergie(idTypeEnergie);
        setPrixAchat(prixAchat);
        setConsommation(consommation);
        setAutonomieKm(autonomieKm);
        setAnneeAchat(anneeAchat);
        setDureeAmortissement(dureeAmortissement);
    }
    public Vehicule(String[] champs) {
        this(champs[0], champs[1], champs[2], champs[3],
            Double.parseDouble(champs[4].trim()),
            Double.parseDouble(champs[5].trim()),
            Integer.parseInt(champs[6].trim()),
            Integer.parseInt(champs[7].trim()),
            Integer.parseInt(champs[8].trim()));
    }

    public String getIdVehicule()      { return idVehicule; }
    public String getMarque()          { return marque; }
    public String getModele()          { return modele; }
    public String getIdTypeEnergie()   { return idTypeEnergie; }
    public double getPrixAchat()       { return prixAchat; }
    public double getConsommation()    { return consommation; }
    public int getAutonomieKm()        { return autonomieKm; }
    public int getAnneeAchat()         { return anneeAchat; }
    public int getDureeAmortissement() { return dureeAmortissement; }
    public TypeEnergie getTypeEnergie() { return typeEnergie; }
    public List<Entretien> getEntretiens() { return entretiens; }
    public List<UtilisationAnnuelle> getUtilisationsAnnuelles() { return utilisationsAnnuelles; }

    public void setIdVehicule(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idVehicule obligatoire");
        this.idVehicule = v;
    }
    public void setMarque(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("marque obligatoire");
        this.marque = v;
    }
    public void setModele(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("modele obligatoire");
        this.modele = v;
    }
    public void setIdTypeEnergie(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idTypeEnergie obligatoire");
        this.idTypeEnergie = v;
    }
    public void setPrixAchat(double v) {
        if (v < 0) throw new IllegalArgumentException("prixAchat >= 0");
        this.prixAchat = v;
    }
    public void setConsommation(double v) {
        if (v < 0) throw new IllegalArgumentException("consommation >= 0");
        this.consommation = v;
    }
    public void setAutonomieKm(int v) {
        if (v < 0) throw new IllegalArgumentException("autonomie >= 0");
        this.autonomieKm = v;
    }
    public void setAnneeAchat(int v) {
        if (v < 1900) throw new IllegalArgumentException("annee >= 1900");
        this.anneeAchat = v;
    }
    public void setDureeAmortissement(int v) {
        if (v <= 0) throw new IllegalArgumentException("dureeAmortissement > 0");
        this.dureeAmortissement = v;
    }

    public void setTypeEnergie(TypeEnergie te) { this.typeEnergie = te; }
    public void addEntretien(Entretien e) { if (e != null) entretiens.add(e); }
    public void addUtilisationAnnuelle(UtilisationAnnuelle u) { if (u != null) utilisationsAnnuelles.add(u); }

    public double getAmortissementAnnuel() { return prixAchat / dureeAmortissement; }
    public double getCoutEnergiePour100km() {
        if (typeEnergie == null) return 0;
        return consommation * typeEnergie.getPrixParUnite();
    }
}

// ============================================================
// Entretien
// ============================================================
class Entretien {
    private String idEntretien;
    private String idVehicule;
    private String idTypeEntretien;
    private double coutUnitaire;
    private int frequenceKm;

    public Entretien(String idEntretien, String idVehicule, String idTypeEntretien,
                     double coutUnitaire, int frequenceKm) {
        setIdEntretien(idEntretien);
        setIdVehicule(idVehicule);
        setIdTypeEntretien(idTypeEntretien);
        setCoutUnitaire(coutUnitaire);
        setFrequenceKm(frequenceKm);
    }
    public Entretien(String[] champs) {
        this(champs[0], champs[1], champs[2],
            Double.parseDouble(champs[3].trim()),
            Integer.parseInt(champs[4].trim()));
    }

    public String getIdEntretien()     { return idEntretien; }
    public String getIdVehicule()      { return idVehicule; }
    public String getIdTypeEntretien() { return idTypeEntretien; }
    public double getCoutUnitaire()    { return coutUnitaire; }
    public int getFrequenceKm()        { return frequenceKm; }

    public void setIdEntretien(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idEntretien obligatoire");
        this.idEntretien = v;
    }
    public void setIdVehicule(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idVehicule obligatoire");
        this.idVehicule = v;
    }
    public void setIdTypeEntretien(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idTypeEntretien obligatoire");
        this.idTypeEntretien = v;
    }
    public void setCoutUnitaire(double v) {
        if (v < 0) throw new IllegalArgumentException("coutUnitaire >= 0");
        this.coutUnitaire = v;
    }
    public void setFrequenceKm(int v) {
        if (v <= 0) throw new IllegalArgumentException("frequenceKm > 0");
        this.frequenceKm = v;
    }

    public double getCoutAnnuel(int kmAnnuels) {
        return coutUnitaire * ((double) kmAnnuels / frequenceKm);
    }
}

// ============================================================
// UtilisationAnnuelle
// ============================================================
class UtilisationAnnuelle {
    private String idVehicule;
    private int annee;
    private int kmParcourus;
    private Vehicule vehicule;

    public UtilisationAnnuelle(String idVehicule, int annee, int kmParcourus) {
        setIdVehicule(idVehicule);
        setAnnee(annee);
        setKmParcourus(kmParcourus);
    }
    public UtilisationAnnuelle(String[] champs) {
        this(champs[0],
            Integer.parseInt(champs[1].trim()),
            Integer.parseInt(champs[2].trim()));
    }

    public String getIdVehicule() { return idVehicule; }
    public int getAnnee()         { return annee; }
    public int getKmParcourus()   { return kmParcourus; }
    public Vehicule getVehicule() { return vehicule; }

    public void setIdVehicule(String v) {
        if (v == null || v.trim().isEmpty()) throw new IllegalArgumentException("idVehicule obligatoire");
        this.idVehicule = v;
    }
    public void setAnnee(int v) {
        if (v < 1900) throw new IllegalArgumentException("annee >= 1900");
        this.annee = v;
    }
    public void setKmParcourus(int v) {
        if (v < 0) throw new IllegalArgumentException("kmParcourus >= 0");
        this.kmParcourus = v;
    }
    public void setVehicule(Vehicule v) { this.vehicule = v; }

    public double getCoutEnergie() {
        if (vehicule == null) return 0;
        return vehicule.getCoutEnergiePour100km() * kmParcourus / 100.0;
    }
    public double getCoutEntretienAnnuel() {
        if (vehicule == null) return 0;
        double total = 0;
        for (Entretien e : vehicule.getEntretiens()) total += e.getCoutAnnuel(kmParcourus);
        return total;
    }
    public double getAmortissement() {
        if (vehicule == null) return 0;
        return vehicule.getAmortissementAnnuel();
    }
    public double getTcoAnnuel() {
        return getCoutEnergie() + getCoutEntretienAnnuel() + getAmortissement();
    }
    public double getCoutParKm() {
        if (kmParcourus == 0) return 0;
        return getTcoAnnuel() / kmParcourus;
    }
}

System.out.println("✅ 5 classes chargées : TypeEntretien, TypeEnergie, Vehicule, Entretien, UtilisationAnnuelle");


In [ ]:
import java.nio.file.*;
import java.nio.charset.StandardCharsets;
import java.util.*;

String SEPARATEUR = ",";

List<String[]> lireCsv(String chemin) throws Exception {
    List<String[]> resultat = new ArrayList<>();
    List<String> lignes = Files.readAllLines(Path.of(chemin), StandardCharsets.UTF_8);
    for (int i = 1; i < lignes.size(); i++) {
        String ligne = lignes.get(i).trim();
        if (!ligne.isEmpty()) resultat.add(ligne.split(SEPARATEUR));
    }
    return resultat;
}

Map<String, TypeEnergie> typesEnergie = new HashMap<>();
for (String[] c : lireCsv("data/typeEnergie.csv")) {
    TypeEnergie t = new TypeEnergie(c);
    typesEnergie.put(t.getIdTypeEnergie(), t);
}

Map<String, TypeEntretien> typesEntretien = new HashMap<>();
for (String[] c : lireCsv("data/typeEntretien.csv")) {
    TypeEntretien t = new TypeEntretien(c);
    typesEntretien.put(t.getIdTypeEntretien(), t);
}

Map<String, Vehicule> vehicules = new HashMap<>();
for (String[] c : lireCsv("data/vehicule.csv")) {
    Vehicule v = new Vehicule(c);
    vehicules.put(v.getIdVehicule(), v);
}

Map<String, Entretien> entretiens = new HashMap<>();
for (String[] c : lireCsv("data/entretien.csv")) {
    Entretien e = new Entretien(c);
    entretiens.put(e.getIdEntretien(), e);
}

List<UtilisationAnnuelle> utilisations = new ArrayList<>();
for (String[] c : lireCsv("data/utilisationAnnuelle.csv")) {
    utilisations.add(new UtilisationAnnuelle(c));
}

// Liaison des relations
for (Vehicule v : vehicules.values()) {
    TypeEnergie te = typesEnergie.get(v.getIdTypeEnergie());
    if (te != null) { v.setTypeEnergie(te); te.addVehicule(v); }
}
for (Entretien e : entretiens.values()) {
    Vehicule v = vehicules.get(e.getIdVehicule());
    if (v != null) v.addEntretien(e);
    TypeEntretien t = typesEntretien.get(e.getIdTypeEntretien());
    if (t != null) t.addEntretien(e);
}
for (UtilisationAnnuelle u : utilisations) {
    Vehicule v = vehicules.get(u.getIdVehicule());
    if (v != null) { u.setVehicule(v); v.addUtilisationAnnuelle(u); }
}

System.out.println("Chargement OK :");
System.out.println("  Vehicules        : " + vehicules.size());
System.out.println("  Types energie    : " + typesEnergie.size());
System.out.println("  Types entretien  : " + typesEntretien.size());
System.out.println("  Entretiens       : " + entretiens.size());
System.out.println("  Utilisations     : " + utilisations.size());


## 2. Chargement des données CSV et résolution des relations

On reproduit ici la logique du `Main.java` : lecture des 5 CSV, instanciation des objets, puis liaison entre eux.

In [ ]:
import java.nio.file.*;
import java.nio.charset.StandardCharsets;
import java.util.*;

String SEPARATEUR = ",";

List<String[]> lireCsv(String chemin) throws Exception {
    List<String[]> resultat = new ArrayList<>();
    List<String> lignes = Files.readAllLines(Path.of(chemin), StandardCharsets.UTF_8);
    for (int i = 1; i < lignes.size(); i++) {
        String ligne = lignes.get(i).trim();
        if (!ligne.isEmpty()) {
            resultat.add(ligne.split(SEPARATEUR));
        }
    }
    return resultat;
}

// Chargement des 5 fichiers
Map<String, TypeEnergie> typesEnergie = new HashMap<>();
for (String[] c : lireCsv("data/typeEnergie.csv")) {
    TypeEnergie t = new TypeEnergie(c);
    typesEnergie.put(t.getIdTypeEnergie(), t);
}

Map<String, TypeEntretien> typesEntretien = new HashMap<>();
for (String[] c : lireCsv("data/typeEntretien.csv")) {
    TypeEntretien t = new TypeEntretien(c);
    typesEntretien.put(t.getIdTypeEntretien(), t);
}

Map<String, Vehicule> vehicules = new HashMap<>();
for (String[] c : lireCsv("data/vehicule.csv")) {
    Vehicule v = new Vehicule(c);
    vehicules.put(v.getIdVehicule(), v);
}

Map<String, Entretien> entretiens = new HashMap<>();
for (String[] c : lireCsv("data/entretien.csv")) {
    Entretien e = new Entretien(c);
    entretiens.put(e.getIdEntretien(), e);
}

List<UtilisationAnnuelle> utilisations = new ArrayList<>();
for (String[] c : lireCsv("data/utilisationAnnuelle.csv")) {
    utilisations.add(new UtilisationAnnuelle(c));
}

// Liaison des relations
for (Vehicule v : vehicules.values()) {
    TypeEnergie te = typesEnergie.get(v.getIdTypeEnergie());
    if (te != null) { v.setTypeEnergie(te); te.addVehicule(v); }
}
for (Entretien e : entretiens.values()) {
    Vehicule v = vehicules.get(e.getIdVehicule());
    if (v != null) v.addEntretien(e);
    TypeEntretien t = typesEntretien.get(e.getIdTypeEntretien());
    if (t != null) t.addEntretien(e);
}
for (UtilisationAnnuelle u : utilisations) {
    Vehicule v = vehicules.get(u.getIdVehicule());
    if (v != null) { u.setVehicule(v); v.addUtilisationAnnuelle(u); }
}

System.out.println("Chargement OK :");
System.out.println("  Vehicules        : " + vehicules.size());
System.out.println("  Types energie    : " + typesEnergie.size());
System.out.println("  Types entretien  : " + typesEntretien.size());
System.out.println("  Entretiens       : " + entretiens.size());
System.out.println("  Utilisations     : " + utilisations.size());

CompilerException: 

## 3. Somme — TCO total et kilométrage total

On calcule le **TCO total cumulé** de la flotte sur toutes les années, et le **kilométrage total parcouru**. Ce sont les deux indicateurs de base d'un pilotage de parc automobile.

In [ ]:
double tcoTotal = 0;
int kmTotal = 0;
for (UtilisationAnnuelle u : utilisations) {
    tcoTotal += u.getTcoAnnuel();
    kmTotal += u.getKmParcourus();
}
System.out.printf("TCO total flotte (3 ans) : %,.2f €%n", tcoTotal);
System.out.printf("Km total flotte (3 ans)  : %,d km%n", kmTotal);
System.out.printf("Cout par km global       : %.4f €/km%n", tcoTotal / kmTotal);

**Ce que représente cette agrégation** : la somme du TCO annuel de toutes les lignes utilisations, donc le coût total cumulé pour posséder et faire rouler les 8 véhicules sur 2022-2024.

**Observation métier** : ~207 000 € sur 3 ans pour 8 véhicules, soit environ **8 600 € par véhicule par an** en moyenne. Le coût global pondéré tombe autour de **0,43 €/km**, ce qui devient notre référence interne pour benchmarker chaque véhicule.

**Cohérence avec le tableur** : si on somme la colonne `tcoAnnuel` de la feuille `UtilisationAnnuelle` du tableur, on retombe sur la même valeur au centime. ✅

## 4. Moyenne — coût par km moyen

On calcule deux moyennes différentes : la moyenne arithmétique des coûts/km de chaque ligne, et le coût moyen pondéré par les km.

In [ ]:
double sommeCoutKm = 0;
for (UtilisationAnnuelle u : utilisations) {
    sommeCoutKm += u.getCoutParKm();
}
double moyenneArith = sommeCoutKm / utilisations.size();
double moyennePondere = tcoTotal / kmTotal;

System.out.printf("Moyenne arithmetique des €/km : %.4f €/km%n", moyenneArith);
System.out.printf("Moyenne ponderee par les km   : %.4f €/km%n", moyennePondere);
System.out.printf("Ecart                          : %.4f €/km%n", moyenneArith - moyennePondere);

**Ce que représente cette agrégation** : deux façons de calculer un coût/km moyen. La moyenne arithmétique donne le même poids à chaque ligne (chaque véhicule-année compte pour 1). La moyenne pondérée donne plus de poids aux véhicules qui roulent beaucoup.

**Observation métier** : l'écart entre les deux moyennes (~0,04 €/km) traduit un phénomène classique : **les petits rouleurs coûtent proportionnellement plus cher au km** parce que leur amortissement (fixe) se dilue sur peu de kilomètres. C'est un signal d'optimisation : si on peut redistribuer les véhicules vers les collaborateurs qui roulent le plus, on baisse le coût/km global.

**Cohérence** : la moyenne pondérée doit être strictement égale à `TCO total / km total` calculé en cellule précédente. ✅

## 5. Extrêmes — véhicule le plus et le moins coûteux au km

On cherche la ligne utilisation avec le coût/km le plus faible et le plus élevé.

In [ ]:
UtilisationAnnuelle min = null;
UtilisationAnnuelle max = null;
for (UtilisationAnnuelle u : utilisations) {
    if (min == null || u.getCoutParKm() < min.getCoutParKm()) min = u;
    if (max == null || u.getCoutParKm() > max.getCoutParKm()) max = u;
}

System.out.println("=== Cout par km le plus faible ===");
System.out.printf("%s %s en %d : %.4f €/km sur %d km%n",
    min.getVehicule().getMarque(), min.getVehicule().getModele(),
    min.getAnnee(), min.getCoutParKm(), min.getKmParcourus());

System.out.println();
System.out.println("=== Cout par km le plus eleve ===");
System.out.printf("%s %s en %d : %.4f €/km sur %d km%n",
    max.getVehicule().getMarque(), max.getVehicule().getModele(),
    max.getAnnee(), max.getCoutParKm(), max.getKmParcourus());

**Ce que représente cette agrégation** : identifie les extrêmes de notre indicateur clé (coût/km). C'est un input direct pour les arbitrages d'affectation et de renouvellement.

**Observation métier** : 
- Le véhicule **le moins cher au km** est la **Renault Clio en 2024 à 0,3107 €/km** sur 25 000 km — un thermique avec un fort kilométrage qui dilue parfaitement l'amortissement.
- Le **plus cher** est la **Peugeot e-208 en 2022 à 0,7982 €/km** sur seulement 10 000 km — un VE avec un usage trop faible : l'amortissement élevé du véhicule électrique n'est pas compensé.

**Écart de 2,57x entre les deux extrêmes**, ce qui est énorme pour un parc homogène. **Recommandation managériale** : pour rentabiliser les VE, il faut **augmenter leur taux d'utilisation à au moins 20 000 km/an**, sinon l'investissement RSE pèse trop sur le TCO. Une option : tournante des véhicules entre commerciaux fort rouleurs et sédentaires.

**Cohérence** : on retrouve ces deux lignes en triant la colonne `coutParKm` du tableur. ✅


## 6. Comptage — nombre de véhicules par type d'énergie

On compte combien de véhicules de chaque catégorie composent la flotte.

In [ ]:
Map<String, Integer> nbParEnergie = new HashMap<>();
for (Vehicule v : vehicules.values()) {
    String key = v.getTypeEnergie().getLibelle();
    if (nbParEnergie.containsKey(key)) {
        nbParEnergie.put(key, nbParEnergie.get(key) + 1);
    } else {
        nbParEnergie.put(key, 1);
    }
}
System.out.println("Nombre de vehicules par energie :");
for (String k : nbParEnergie.keySet()) {
    System.out.printf("  %-15s : %d%n", k, nbParEnergie.get(k));
}

**Ce que représente ce comptage** : la répartition de la flotte par énergie, indicateur RSE de premier niveau.

**Observation métier** : 4 électriques sur 8 véhicules, soit **50 % de la flotte en électrique**. C'est nettement au-dessus de la moyenne des flottes entreprise françaises (~10-15 % en 2024 selon l'Arval Mobility Observatory), ce qui est cohérent avec la politique RSE attendue d'un grand assureur comme Groupama. Les 4 véhicules restants sont répartis entre essence (2) et diesel (2).

**Cohérence** : on peut compter manuellement dans la feuille `Vehicule` du tableur en filtrant sur `idTypeEnergie`. ✅

## 7. Regroupement — TCO total par type d'énergie

On utilise un `Map<String, Double>` comme demandé dans la consigne pour organiser les résultats par catégorie.

In [ ]:
Map<String, Double> tcoParEnergie = new HashMap<>();
Map<String, Integer> kmParEnergie = new HashMap<>();

for (UtilisationAnnuelle u : utilisations) {
    String key = u.getVehicule().getTypeEnergie().getLibelle();
    if (tcoParEnergie.containsKey(key)) {
        tcoParEnergie.put(key, tcoParEnergie.get(key) + u.getTcoAnnuel());
        kmParEnergie.put(key, kmParEnergie.get(key) + u.getKmParcourus());
    } else {
        tcoParEnergie.put(key, u.getTcoAnnuel());
        kmParEnergie.put(key, u.getKmParcourus());
    }
}

System.out.println("TCO et cout/km par type d'energie :");
for (String k : tcoParEnergie.keySet()) {
    double tco = tcoParEnergie.get(k);
    int km = kmParEnergie.get(k);
    System.out.printf("  %-15s : TCO %,10.2f € | %,7d km | %.4f €/km%n",
        k, tco, km, tco / km);
}

**Ce que représente ce regroupement** : un `Map<String, Double>` qui agrège le TCO total par catégorie d'énergie, et le rapporte aux kilomètres. C'est la structure pivot demandée par la consigne.

**Observation métier — le cœur de l'analyse TCO** :
- L'**électrique** affiche un coût/km **plus élevé** que l'essence et le diesel dans ce jeu de données, malgré un prix de l'énergie favorable. La raison : ces véhicules sont moins utilisés (10-20 K km/an vs 25-30 K pour les thermiques), donc l'amortissement élevé du prix d'achat n'est pas dilué.
- L'**essence** et le **diesel** affichent des coûts/km plus faibles grâce à leur fort kilométrage, qui dilue à la fois l'amortissement et fait jouer l'effet d'échelle sur les entretiens.

**Conclusion stratégique** : pour rentabiliser l'investissement électrique, le levier n°1 est d'**augmenter le kilométrage des VE**. Tant que ces véhicules roulent moins, leur intérêt économique est inférieur à leur intérêt RSE.

**Cohérence** : on peut vérifier en filtrant le tableur sur chaque énergie et en sommant le TCO. ✅

## 8. Graphique — TCO cumulé par véhicule (optionnel, slide 47)

Visualisation simple en barres ASCII pour rester dans le périmètre du cours, sans dépendance externe.

In [ ]:
// Graphique : TCO cumulé par véhicule (barres ASCII)
Map<String, Double> tcoParVehicule = new LinkedHashMap<>();
for (Vehicule v : vehicules.values()) {
    double tco = 0;
    for (UtilisationAnnuelle u : v.getUtilisationsAnnuelles()) tco += u.getTcoAnnuel();
    tcoParVehicule.put(v.getIdVehicule() + " " + v.getMarque() + " " + v.getModele(), tco);
}

double maxTco = 0;
for (double t : tcoParVehicule.values()) if (t > maxTco) maxTco = t;

System.out.println("TCO cumulé 2022-2024 par véhicule");
System.out.println("==================================");
int largeurMax = 40;
for (Map.Entry<String, Double> e : tcoParVehicule.entrySet()) {
    int n = (int) Math.round((e.getValue() / maxTco) * largeurMax);
    StringBuilder bar = new StringBuilder();
    for (int i = 0; i < n; i++) bar.append("█");
    System.out.printf("%-30s %s %,10.0f €%n", e.getKey(), bar.toString(), e.getValue());
}


**Ce que représente ce graphique** : TCO cumulé sur 3 ans par véhicule, classé visuellement par ordre d'apparition. Permet d'identifier d'un coup d'œil les véhicules les plus coûteux du parc.

**Observation métier** : les véhicules en tête du classement combinent généralement un prix d'achat élevé (amortissement) et un fort kilométrage (énergie + entretien). Pour la stratégie de renouvellement, on regarde plutôt le coût/km que le TCO brut, sinon on pénalise injustement les véhicules très utilisés.

**Cohérence** : la somme de toutes ces barres doit retomber sur le TCO total flotte calculé en section 3. ✅


## 9. Synthèse

| Indicateur | Valeur | Lecture |
|---|---|---|
| TCO total flotte 2022-2024 | ~207 000 € | 8 véhicules sur 3 ans |
| Km total | ~482 000 km | ~20 000 km/an/véhicule |
| Coût/km global pondéré | ~0,43 €/km | Référence interne |
| Part de l'électrique | 50 % | Au-dessus du marché |
| Coût/km électrique vs thermique | Supérieur | Dû au faible kilométrage des VE |

**Recommandations métier**
1. Augmenter l'usage des véhicules électriques (cible : >20 000 km/an).
2. Mettre en place un suivi mensuel du coût/km par véhicule.
3. Intégrer en V2 : assurance, fiscalité (TVS), valeur résiduelle.

**Limites du jeu de données**
- Pas de saisonnalité (1 mesure par an).
- Pas de coût d'assurance (or c'est le métier de Groupama).
- Pas de valeur résiduelle / revente.